# TF-IDF + SGDClassifier (modified huber)
**text › classification** · notebook `tfidf_sgd`

**Kapan dipakai:** Data teks sangat besar (jutaan baris) atau butuh pelatihan bertahap (partial_fit).

**Jebakan umum:** Sensitif terhadap alpha dan jumlah epoch; hasil bervariasi antar seed. Skala fitur sudah L2 dari TF-IDF.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/text/classification/emot_train.csv"
TEST_PATH  = ROOT / "data_latihan/text/classification/emot_test.csv"      # None kalau tidak ada file test
TEXT_COL   = "text"
TARGET     = "label"
ID_COL     = 'id'
LANG       = "id"                   # "id" Indonesia, "en" Inggris
CLEAN      = True                          # bersihkan URL/@mention/huruf berulang
USE_STOPWORDS = False                      # stopword sering merugikan sentimen (menghapus negasi); coba True untuk topik
SCORING    = "f1_macro"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "tfidf_sgd"
OUT = out_dir("text", "classification", MODEL_KEY)
set_seed(SEED)

## 2. Baca data & lihat sekilas

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
print("teks kosong:", int(df[TEXT_COL].isna().sum()), "| duplikat teks:", int(df[TEXT_COL].duplicated().sum()))
display(df[TARGET].value_counts(normalize=True).round(3).to_frame("proporsi"))
L = df[TEXT_COL].fillna("").str.split().str.len()
print(f"panjang (kata): median={L.median():.0f}, p95={L.quantile(.95):.0f}, maks={L.max()}")
display(df.sample(3, random_state=SEED)[[TEXT_COL, TARGET]])

## 3. Bersihkan & pisahkan
Split dilakukan SEBELUM vectorizer dilatih, jadi kosakata hanya dari data training.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from superdb.text import clean_text, ID_STOPWORDS, top_terms, show_errors
from superdb.tabular import cv_score
prep = (lambda s: s.map(clean_text)) if CLEAN else (lambda s: s.fillna("").astype(str))
X = prep(df[TEXT_COL]); le = LabelEncoder(); y = le.fit_transform(df[TARGET]); CLASSES = list(le.classes_)
print("kelas:", dict(enumerate(CLASSES)))
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=VALID_SIZE, random_state=SEED, stratify=y)
STOP = (ID_STOPWORDS if LANG == "id" else "english") if USE_STOPWORDS else None
print(len(Xtr), "latih /", len(Xva), "validasi")

## 4. Model
Kode `vec` dan `model` di bawah bisa disalin langsung.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, strip_accents="unicode", stop_words=STOP)
model = SGDClassifier(loss='modified_huber', alpha=2e-5, max_iter=50, random_state=42)
pipe = Pipeline([("vec", vec), ("model", model)])
pipe

## 5. Cross-validation (pada data training)

In [ ]:
folds, summary = cv_score(pipe, Xtr, ytr, "classification", SCORING, N_SPLITS, SEED)
display(folds.round(4)); print(summary)

## 6. Latih & evaluasi pada validation

In [ ]:
pipe.fit(Xtr, ytr)
pred = pipe.predict(Xva)
proba = pipe.predict_proba(Xva) if hasattr(pipe, "predict_proba") else None
metrics = classification_metrics(yva, pred, proba, classes=np.arange(len(CLASSES)))
display(show_table(metrics))
from sklearn.metrics import classification_report
print(classification_report(yva, pred, target_names=[str(c) for c in CLASSES]))
plot_confusion(yva, pred, labels=np.arange(len(CLASSES)), path=OUT / "confusion.png"); plt.show()

## 7. Kata/n-gram paling khas tiap kelas
Bahan analisis untuk laporan. Hanya untuk model linear / Naive Bayes.

In [ ]:
try:
    tt = top_terms(pipe, CLASSES, n=10); display(tt); tt.to_csv(OUT / "top_terms.csv", index=False)
except Exception as ex:
    print("kata penting tidak tersedia untuk model ini:", type(ex).__name__, ex)

## 8. Analisis error
Baca contoh yang salah: pola kesalahan (sarkasme, label ambigu, kelas bertetangga) lebih berguna daripada satu angka metrik.

In [ ]:
display(show_errors(Xva.tolist(), yva, pred, CLASSES, n=8))

## 9. Latih ulang pada semua data → prediksi test → submission

In [ ]:
from sklearn.base import clone
final = clone(pipe).fit(X, y)
if test is not None:
    Xte = prep(test[TEXT_COL])
    ids = test[ID_COL] if ID_COL else np.arange(len(test))
    sub = make_submission(ids, le.inverse_transform(final.predict(Xte)), ID_COL or "id", TARGET, OUT / "submission.csv")
    display(sub.head(3)); display(sub[TARGET].value_counts(normalize=True).round(3).to_frame("proporsi prediksi"))
save_model({"pipe": final, "classes": CLASSES, "clean": CLEAN}, OUT / "model.joblib")
save_json({"cv": summary, "valid": metrics, "model": MODEL_KEY}, OUT / "metrics.json")
b = load_model(OUT / "model.joblib")
assert (b["pipe"].predict(X.head(50)) == final.predict(X.head(50))).all(), "model muat-ulang beda"
print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan beberapa model di folder ini. Pilih berdasarkan **CV**, bukan satu split.
- Skor CV jauh di atas test? Cek duplikat teks antar-split dan pergeseran domain (`notebooks/tabular/validation/adversarial_validation`).
- Ingin lebih tinggi: embedding kalimat atau fine-tuning transformer (folder yang sama, bertanda *heavy*), lalu blend dengan model TF-IDF.
- Teks panjang/berbahasa Indonesia: stemming Sastrawi opsional (`cookbook/`).